In [1]:
import numpy as np

In [ ]:
def sigmoid(x):
    # activation function f(x) = 1/ (1 + e ^(-x))
    return 1/(1 + np.exp(-x))

def der_sigmoid(x):
    # deration of sigmoid: f'(x) = f(x) * (1 - f(x))
    fx = sigmoid(x)
    return fx * (1 - fx)

def mse_loss(true_y, pred_y):
    # both are numpy array with the same length
    return np.square(true_y - pred_y).mean()

In [14]:
class SimpleNN:
    '''
    A simple neuro network with:
    - 2 inputs 
    - a hidden layer with 2 neurons (h1, h2)
    - an output layer with 1 neurons o1
    '''

    def __init__(self):
        # input weights for first neuron h1
        self.w1 = np.random.normal()
        self.w2 = np.random.normal()

        # input weights for second neuron h2
        self.w3 = np.random.normal()
        self.w4 = np.random.normal()
        
        # input weights for output neuron o1
        self.w5 = np.random.normal()
        self.w6 = np.random.normal()

        # bias for h1, h2 and o1
        self.b1 = np.random.normal()
        self.b2 = np.random.normal()
        self.b3 = np.random.normal()

    def feedforward(self, x):
        # x is 1D array with length = 2
        h1 = sigmoid(self.w1 * x[0] + self.w2 * x[1] + self.b1)
        h2 = sigmoid(self.w3 * x[0] + self.w4 * x[1] + self.b2)
        o1 = sigmoid(self.w5 * h1 + self.w6 * h2 + self.b3)
        return o1
    
    def train(self, train_data, validations):
        '''
        train_data is a [n x 2] array, where n is number of samples in dataset
        validations is a 1D array, with length n, contains true values correspond to those in data.
        '''
        learn_rate = 0.15
        epochs = 1000
        
        for epoch in range(epochs):
            for x, true_y in zip(train_data, validations):
                sum_h1 = self.w1 * x[0] + self.w2 * x[1] + self.b1
                h1 = sigmoid(sum_h1)

                sum_h2 = self.w3 * x[0] + self.w4 * x[1] + self.b2
                h2 = sigmoid(sum_h2)

                sum_o1 = self.w5 * h1 + self.w6 * h2 + self.b3
                o1 = sigmoid(sum_o1)
                pred_y = o1

                # --- partial deratives (back prop)

                d_L_d_pred_y = -2 * (true_y - pred_y)

                # derivatives of o1
                d_pred_y_d_w5 = h1 * der_sigmoid(sum_o1)
                d_pred_y_d_w6 = h2 * der_sigmoid(sum_o1)
                d_pred_y_d_b3 = der_sigmoid(sum_o1)

                d_pred_y_d_h1 = self.w5 * der_sigmoid(sum_o1)
                d_pred_y_d_h2 = self.w6 * der_sigmoid(sum_o1)

                # derivatives of h1
                d_h1_d_w1 = x[0] * der_sigmoid(sum_h1)
                d_h1_d_w2 = x[1] * der_sigmoid(sum_h1)
                d_h1_d_b1 = der_sigmoid(sum_h1)

                # derivatives of h2
                d_h2_d_w3 = x[0] * der_sigmoid(sum_h2)
                d_h2_d_w4 = x[1] * der_sigmoid(sum_h2)
                d_h2_d_b2 = der_sigmoid(sum_h2)

                # --- update weights and biases
                # update for h1
                self.w1 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h1 * d_h1_d_w1
                self.w2 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h1 * d_h1_d_w2
                self.b1 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h1 * d_h1_d_b1

                # update for h2
                self.w3 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h2 * d_h2_d_w3
                self.w4 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h2 * d_h2_d_w4
                self.b2 -= learn_rate * d_L_d_pred_y * d_pred_y_d_h2 * d_h2_d_b2

                # update for o1
                self.w5 -= learn_rate * d_L_d_pred_y * d_pred_y_d_w5
                self.w6 -= learn_rate * d_L_d_pred_y * d_pred_y_d_w6
                self.b3 -= learn_rate * d_L_d_pred_y * d_pred_y_d_b3

                # --- Calculate epouch loss
                if epoch %10 == 0:
                    pred_ys = np.apply_along_axis(self.feedforward, 1, train_data)
                    loss = mse_loss(validations, pred_ys)
                    print(f"Epoch {epoch} loss: {loss:.5f}")


In [6]:
# Define dataset
data = np.array([
  [-2, -1],  # Alice
  [25, 6],   # Bob
  [17, 4],   # Charlie
  [-15, -6], # Diana
])
all_y_trues = np.array([
  1, # Alice
  0, # Bob
  0, # Charlie
  1, # Diana
])

In [ ]:
network = SimpleNN()
network.train(data, all_y_trues)

In [16]:
# Make some predictions
emily = np.array([-7, -3]) # 128 pounds, 63 inches
frank = np.array([20, 2])  # 155 pounds, 68 inches
print("Emily: %.3f" % network.feedforward(emily)) # 0.951 - F
print("Frank: %.3f" % network.feedforward(frank)) # 0.039 - M

Emily: 0.959
Frank: 0.031
